# 关于本 Notebook

本 Notebook 介绍**大型工具生态（large tool ecosystems）**，以及智能体如何在**规模大、异构的工具集合（large, heterogeneous tool sets）**中进行推理。

与为每个集成编写固定代码不同，本 Notebook 会根据自然语言用例在 Composio 中搜索匹配工具，再把这些工具暴露给智能体。

重点不在执行性能，而在**工具发现（Tool Discovery）**、**规划（Planning）**、**认证（Authentication）**与**编排（Orchestration）**。

---

## 你在练习的核心思想

本 Notebook 展示智能体如何从：

* “我只有一个工具” <br>
→ 转变为
* “我有数百个工具，该如何决定用哪个？”

关键概念包括：

* 通过 Composio 进行**工具发现（Tool Discovery）**
* 基于自然语言用例进行**工具选择（Tool Selection）**
* 执行前完成**工作流规划（Workflow Planning）**
* 为需要认证的工具进行**连接管理（Connection Management）**
* **多轮工具推理（Multi-turn Tool Reasoning）**

场景生成器（`generate_scenarios`）连接了：
* Composio 的元工具 schema
* 与真实的智能体任务

---

如果只记住一点：

**现代智能体不只是调用工具。  
它们会动态地发现、规划、认证并组合工具。**

Composio 就是支持跨数百种应用工具包实现这一能力的基础设施层之一。

In [ ]:
!pip install composio openai-agents

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.2/276.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.1/856.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
  Created wheel for pysher: filename=Pysher-1.0.8-py3-none-any.whl size=9889 sha256=6c5a87c046a5f6d6e6b2b5dce1cbdc96a2f1c1e54a0492778401342e288c8eca
  Stored in directory: /root/.cache/pip/wheels/50/f5/b7/7d1662a18f42dddaf2fc5ad7f8611ef264cf0f634f376c3e6c
Successfully built pysher


# Timer

In [ ]:
SET_TIMER = False  # False, True, or minutes as a number

import requests, types
url = "https://raw.githubusercontent.com/Nicolepcx/ORM-self-improving-ai-agents-course/main/timer.py"

timer = types.ModuleType("timer")
exec(requests.get(url).text, timer.__dict__)

timer.start_exam_timer(enabled=SET_TIMER, minutes=15, warn_minutes=5)

## 安装（Installation）


In [ ]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install openpipe-art[backend]==0.4.11 tenacity composio composio_openai openai --prerelease allow --no-cache-dir
else:
    try:
        import numpy
        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    get_vllm, get_triton = (
        ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    )
    !uv pip install --upgrade \
        openpipe-art[backend]==0.4.11 tenacity composio composio_openai openai pillow==11.3.0 protobuf==5.29.5 {get_vllm} {get_numpy} --prerelease allow --no-cache-dir
    !uv pip install -qqq {get_triton}


# 设置 API Key

<font color="red" size="5">
<b>运行本 Notebook 前请注意</b>
</font>
<br>

你需要一个 `OPENROUTER_API_KEY`（[在此获取](https://openrouter.ai/)），以及来自 [Composio](https://platform.composio.dev/) 的 `COMPOSIO_API_KEY`。

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
COMPOSIO_API_KEY = os.getenv("COMPOSIO_API_KEY")

# 初始化后续可能会使用的变量
composio_scenarios = []



# 导入依赖

In [ ]:
import json
import random
import time
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

import art
from art.trajectories import Trajectory, Choice, TrajectoryGroup
from art.mcp import generate_scenarios
from art.mcp.generate_scenarios import preview_scenarios
from art.utils.logging import info, ok, step, warn, err

from openai import OpenAI
from composio import Composio
from composio_openai import OpenAIProvider





# 注意：这里承接 Chapter 3 的 ART 与 RULER，并将其应用到 MCP

## 什么是智能体轨迹（Agent Trajectories）？

在 ART 中，一条**轨迹（trajectory）**表示智能体与环境之间的一次完整交互序列：

- **消息（Messages）**：用户输入、系统 Prompt、智能体响应
- **工具调用（Tool Calls）**：携带参数的函数调用
- **工具结果（Tool Results）**：工具执行后的返回结果
- **奖励（Rewards）**：来自 RULER 或其他评估器的评分
- **指标（Metrics）**：交互过程的元数据，例如轮数、是否成功等

系统会收集并分组多条轨迹，然后由 RULER 进行**相对排序（relative ranking）**，判断哪些轨迹表现更好。这样，模型可以从轨迹之间的比较中学习，而不是依赖单一的绝对分数。

### 使用 Composio 集成更丰富的工具

**为什么多工具很重要：**
- 不同应用工具包各有所长，例如邮件、日历、GitHub、Slack、网页搜索等
- 组合多个工具，可以构建比单次 API 调用更贴近真实业务的工作流
- 一个能力成熟的智能体，应当能够跨多个服务完成发现、规划、认证和执行

**本 Notebook 中的 Composio 流程：**
- 使用 `composio.tools.get(...)` 获取与 `USE_CASE` 匹配的工具
- 将这些工具传入兼容 OpenAI 的工具调用循环
- 由 Composio 通过其 provider 执行被选中的工具调用
- 使用生成的场景探索不同工作流与难度等级

**可以思考：**
- 哪些工具包与你的用例最相关？
- 面对不同类型的任务，智能体应如何决定使用哪个工具？
- 真正执行之前，需要完成哪些连接或认证步骤？

In [ ]:
# Composio 配置
# Composio 提供用于发现、认证和执行工具的元工具
# 运行示例前，请先在环境变量或 .env 文件中设置 COMPOSIO_API_KEY

In [ ]:
# @title Composio 配置

# Composio 配置
# 1. 在 https://platform.composio.dev/ 创建 API key
# 2. 在环境变量或 .env 文件中将其设置为 COMPOSIO_API_KEY
# 3. 为本次 Notebook 运行选择一个稳定的 user_id

COMPOSIO_USER_ID = os.getenv("COMPOSIO_USER_ID", "course-user@example.com")
COMPOSIO_API_KEY = os.getenv("COMPOSIO_API_KEY") or os.environ.get("COMPOSIO_API_KEY", "")
COMPOSIO_TOOL_LIMIT = 10
COMPOSIO_FALLBACK_TOOLKITS = ["GMAIL", "OUTLOOK", "SENDGRID"]
COMPOSIO_FALLBACK_TOOLS = [
    "GMAIL_SEND_EMAIL",
    "GMAIL_CREATE_EMAIL_DRAFT",
    "GMAIL_SEND_DRAFT",
    "OUTLOOK_SEND_EMAIL",
    "SENDGRID_SEND_EMAIL_WITH_TWILIO_SEND_GRID",
]

if not COMPOSIO_API_KEY:
    warn("COMPOSIO_API_KEY not set. Please:")
    warn("1. Visit https://platform.composio.dev/")
    warn("2. Create an API key")
    warn("3. Set it in your .env file as COMPOSIO_API_KEY or export COMPOSIO_API_KEY=your_key")
    composio = None
else:
    # 本 Notebook 使用 Composio quick start 中直接调用 tools.get(...) 的方式
    composio = Composio(api_key=COMPOSIO_API_KEY, provider=OpenAIProvider())
    ok("Composio configured")
    print(f"User ID: {COMPOSIO_USER_ID}")
    print("API key: [configured]")


[09:23:45] OK    Composio configured
User ID: course-user@example.com
API key: [configured]


In [ ]:
NUM_SCENARIOS = 5 # 演示时使用较小数值
MAX_TURNS = 5 # 演示时使用较小数值
USE_CASE = "I want to send an email. Show me what tools are available and what the workflow would look like."
LLM_MODEL = "openai/gpt-5.4-mini"

In [ ]:
# @title Composio 辅助函数


def _tool_function(tool: Any) -> Dict[str, Any]:
    """返回 dict 或 SDK model 工具对应的 function payload。"""
    if hasattr(tool, "model_dump"):
        tool = tool.model_dump()
    elif hasattr(tool, "dict"):
        tool = tool.dict()

    if not isinstance(tool, dict):
        return {}
    if "function" in tool and isinstance(tool["function"], dict):
        return tool["function"]
    if tool.get("type") == "function" and "name" in tool:
        # 某些 provider 使用 Responses API 形态：{type, name, description, parameters}
        return tool
    return tool


def _clean_json_schema(schema: Any) -> Dict[str, Any]:
    """将工具参数限制在 Chat Completions 接受的 JSON Schema 子集中。"""
    if not isinstance(schema, dict):
        return {"type": "object", "properties": {}}

    allowed = {
        "type", "properties", "required", "description", "enum", "items", "anyOf",
        "oneOf", "allOf", "default", "additionalProperties", "format", "title",
        "minimum", "maximum", "minLength", "maxLength", "minItems", "maxItems",
    }
    cleaned = {}
    for key, value in schema.items():
        if key not in allowed:
            continue
        if key == "properties" and isinstance(value, dict):
            cleaned[key] = {name: _clean_json_schema(prop) for name, prop in value.items()}
        elif key in {"items", "additionalProperties"} and isinstance(value, dict):
            cleaned[key] = _clean_json_schema(value)
        elif key in {"anyOf", "oneOf", "allOf"} and isinstance(value, list):
            cleaned[key] = [_clean_json_schema(item) for item in value]
        else:
            cleaned[key] = value

    cleaned.setdefault("type", "object")
    if cleaned.get("type") == "object":
        cleaned.setdefault("properties", {})
    return cleaned


def composio_tool_to_art_info(tool: Any) -> Dict[str, Any]:
    """将 Composio/OpenAI 工具转换为 ART 所期望的 schema 结构。"""
    fn = _tool_function(tool)
    return {
        "name": fn.get("name", "UNKNOWN_TOOL"),
        "description": fn.get("description", ""),
        "parameters": _clean_json_schema(fn.get("parameters", {"type": "object", "properties": {}})),
    }


def composio_tool_to_openai_chat_tool(tool: Any) -> Dict[str, Any]:
    """将 Composio 工具规范化为 Chat Completions/OpenRouter 要求的严格工具结构。"""
    fn = composio_tool_to_art_info(tool)
    return {
        "type": "function",
        "function": {
            "name": fn["name"],
            "description": fn["description"],
            "parameters": fn["parameters"],
        },
    }


def _short_json(value: Any, max_chars: int = 500) -> str:
    text = json.dumps(value, indent=2, default=str) if not isinstance(value, str) else value
    return text if len(text) <= max_chars else text[:max_chars] + "... [truncated]"


def _get_composio_tools(**kwargs):
    """兼容不同 SDK 版本中 user_id 参数位置差异，并调用 tools.get。"""
    try:
        return composio.tools.get(user_id=COMPOSIO_USER_ID, **kwargs)
    except TypeError:
        return composio.tools.get(COMPOSIO_USER_ID, **kwargs)


def get_composio_tools_for_use_case(use_case: str, limit: int = COMPOSIO_TOOL_LIMIT):
    """获取 Composio 工具；若语义搜索没有返回工具，则依次尝试回退策略。"""
    if composio is None:
        warn("Composio not configured - skipping Composio examples")
        return []

    lookup_attempts = [
        ("semantic search", {"search": use_case, "limit": limit}),
        ("semantic search, latest versions", {"search": use_case, "limit": limit, "toolkit_versions": "latest"}),
        ("exact email tool slugs", {"tools": COMPOSIO_FALLBACK_TOOLS}),
        ("exact email tool slugs, latest versions", {"tools": COMPOSIO_FALLBACK_TOOLS, "toolkit_versions": "latest"}),
        ("email toolkits", {"toolkits": COMPOSIO_FALLBACK_TOOLKITS, "limit": limit}),
        ("email toolkits lowercase", {"toolkits": [t.lower() for t in COMPOSIO_FALLBACK_TOOLKITS], "limit": limit}),
        ("email toolkits, latest versions", {"toolkits": COMPOSIO_FALLBACK_TOOLKITS, "limit": limit, "toolkit_versions": "latest"}),
    ]

    last_error = None
    for label, kwargs in lookup_attempts:
        try:
            tools = _get_composio_tools(**kwargs)
        except TypeError as e:
            last_error = e
            continue
        except Exception as e:
            last_error = e
            warn(f"Composio lookup failed for {label}: {e}")
            continue

        if tools:
            ok(f"Loaded {len(tools)} Composio tool(s) via {label}")
            return tools

        info(f"No tools returned via {label}; trying next lookup strategy.")

    if last_error:
        warn(f"No Composio tools found. Last lookup error: {last_error}")
    else:
        warn("No Composio tools found after all lookup strategies.")
    return []


async def list_composio_tools_and_resources():
    """获取智能体针对当前 USE_CASE 所使用的 Composio 工具。"""
    tools = get_composio_tools_for_use_case(USE_CASE)
    resources = []
    return tools, resources


if composio is not None:
    ok("Composio helpers configured")
else:
    warn("Composio not configured - skipping Composio examples")


[09:23:45] OK    Composio helpers configured


In [ ]:
# @title OpenRouter + Composio：完整工作流示例

async def list_composio_tools_example(tools: List[Any]):
    """示例：列出为当前用例选中的 Composio 工具。"""
    tool_infos = [composio_tool_to_art_info(tool) for tool in tools]
    print(f"📋 Found {len(tool_infos)} Composio tools for this use case:\n")

    tools_by_toolkit = {}
    for tool in tool_infos:
        toolkit = tool["name"].split("_")[0] if "_" in tool["name"] else "OTHER"
        tools_by_toolkit.setdefault(toolkit, []).append(tool)

    for toolkit, toolkit_tools in tools_by_toolkit.items():
        print(f"  {toolkit.title()}:")
        for tool in toolkit_tools:
            print(f"    - {tool['name']}")
            if tool["description"]:
                desc = tool["description"].split("\n")[0][:100]
                print(f"      {desc}...")
        print()

    return tool_infos


async def composio_workflow_example(use_case: str):
    """示例：为某个用例搜索 Composio 工具，然后让模型使用这些工具。"""
    tools = get_composio_tools_for_use_case(use_case)
    if not tools:
        warn("No Composio tools available")
        return ""

    # Composio 可能返回 provider 特定的工具结构；OpenRouter 的 Chat
    # Completions endpoint 要求严格的 {type: function, function: ...} 结构。
    chat_tools = [composio_tool_to_openai_chat_tool(tool) for tool in tools]
    chat_tools = [tool for tool in chat_tools if tool["function"]["name"] != "UNKNOWN_TOOL"]
    if not chat_tools:
        warn("No Chat Completions-compatible Composio tools available")
        return ""

    llm = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

    tool_names = [tool["function"]["name"] for tool in chat_tools]
    system_prompt = f"""You are an AI assistant with access to Composio tools for workflow automation.

The notebook has already searched Composio for tools relevant to the user's use case and exposed only those tools to you.

Available tool names:
{json.dumps(tool_names, indent=2)}

Workflow pattern:
1. Explain which available tools match the user's goal.
2. If a tool requires account authentication and execution fails or is not possible, explain which app connection is needed.
3. Only execute a tool when the user has provided enough concrete inputs.
4. If the user is only asking to discover tools or see a workflow, summarize the tools and recommended workflow instead of performing a real-world action.
"""

    messages: list[dict[str, Any]] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": use_case},
    ]

    print(f"🚀 Starting Composio workflow for: {use_case}\n")

    for turn in range(MAX_TURNS):
        response = llm.chat.completions.create(
            model="openai/o4-mini",
            messages=messages,
            tools=chat_tools,
            tool_choice="auto" if chat_tools else None,
        )

        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if msg.content:
            print(f"💬 Assistant: {msg.content}\n")

        if not msg.tool_calls:
            break

        print(f"🔧 Tool calls ({len(msg.tool_calls)}):")
        for tc in msg.tool_calls:
            tool_args = json.loads(tc.function.arguments or "{}")
            print(f"  - {tc.function.name}")
            print(f"    Args: {_short_json(tool_args, max_chars=300)}")

        try:
            results = composio.provider.handle_tool_calls(
                response=response,
                user_id=COMPOSIO_USER_ID,
            )
        except Exception as e:
            # 对发现/规划类演示，即使账户尚未连接，或 SDK 无法执行规范化调用，
            # 也保持 Notebook 继续运行。
            results = [
                {
                    "successful": False,
                    "error": str(e),
                    "note": "Tool call was generated, but execution was skipped or failed. Connect the required app account before real execution.",
                }
                for _ in msg.tool_calls
            ]

        for tc, result in zip(msg.tool_calls, results):
            print(f"    Result: {_short_json(result, max_chars=500)}\n")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, default=str),
            })

    return messages[-1].get("content", "") if isinstance(messages[-1], dict) else ""


# 示例 1：列出当前用例可用的 Composio 工具
if composio is not None:
    print("=" * 60)
    print("Example 1: Listing Composio Tools")
    print("=" * 60)
    try:
        tools, _ = await list_composio_tools_and_resources()
        tool_infos = await list_composio_tools_example(tools)
        ok(f"Successfully listed {len(tool_infos)} tools")
    except Exception as e:
        warn(f"Failed to list tools: {e}")
else:
    warn("Composio not configured. Please set COMPOSIO_API_KEY.")


# 示例 2：完整工作流（需要 OpenRouter API key）
if composio is not None and OPENROUTER_API_KEY:
    print("\n" + "=" * 60)
    print("Example 2: Composio Workflow - Search for Tools")
    print("=" * 60)

    try:
        result = await composio_workflow_example(use_case=USE_CASE)
        print("\n✅ Workflow completed!")
    except Exception as e:
        warn(f"Workflow failed: {e}")
else:
    info("\n💡 To run the full workflow example:")
    info("   1. Set COMPOSIO_API_KEY in your .env file")
    info("   2. Set OPENROUTER_API_KEY in your .env file")
    info("   3. Re-run this cell")


Example 1: Listing Composio Tools
[09:23:47] INFO  No tools returned via semantic search; trying next lookup strategy.
[09:23:47] OK    Loaded 4 Composio tool(s) via exact email tool slugs
📋 Found 4 Composio tools for this use case:

  Gmail:
    - GMAIL_CREATE_EMAIL_DRAFT
      Creates a gmail email draft, supporting to/cc/bcc, subject, plain/html body (ensure `is html=true` f...
    - GMAIL_SEND_DRAFT
      Sends the specified, existing draft to the recipients in the to, cc, and bcc headers....
    - GMAIL_SEND_EMAIL
      Sends an email via gmail api using the authenticated user's google profile display name, requiring `...

  Sendgrid:
    - SENDGRID_SEND_EMAIL_WITH_TWILIO_SEND_GRID
      The mail send operation uses sendgrid's v3 api to send emails. visit the provided link for an overvi...

[09:23:47] OK    Successfully listed 4 tools

Example 2: Composio Workflow - Search for Tools
[09:23:49] INFO  No tools returned via semantic search; trying next lookup strategy.
[09:23:49] OK 

In [ ]:
# @title 使用 Composio 工具生成场景

# 简单示例：针对 USE_CASE 选中的一个 Composio 工具生成场景

if composio is not None and OPENROUTER_API_KEY:
    # 第 1 步：通过 Composio 的语义工具搜索获取可用工具
    tools_result, resources_result = await list_composio_tools_and_resources()
    tool_infos = [composio_tool_to_art_info(tool) for tool in tools_result]

    # 第 2 步：选择第一个匹配工具，用于聚焦演示
    selected_tool = tool_infos[0] if tool_infos else None

    if selected_tool:
        selected_tool_name = selected_tool["name"]
        info(f"Selected tool: {selected_tool_name}")

        # 第 3 步：生成场景（与其他工具示例类似）
        try:
            scenario_collection = await generate_scenarios(
                tools=[selected_tool],
                resources=[],
                num_scenarios=NUM_SCENARIOS,
                show_preview=True,
                generator_model=LLM_MODEL,
                generator_api_key=OPENROUTER_API_KEY,
            )

            scenarios = [{"task": s.task, "difficulty": s.difficulty} for s in scenario_collection.scenarios]
            composio_scenarios.extend(scenarios)
            ok(f"Generated {len(scenarios)} scenarios for {selected_tool_name}")

            info("Sample scenarios:")
            preview_scenarios(scenarios, n=min(3, len(scenarios)))

        except Exception as e:
            warn(f"Scenario generation failed: {e}")
    else:
        warn("No matching Composio tools found for this USE_CASE after all fallback lookups.")
        warn("If you do not see lookup-strategy messages above, rerun the Composio Helper Functions cell first.")
else:
    warn("Composio or OpenRouter not configured. Please set COMPOSIO_API_KEY and OPENROUTER_API_KEY.")

[09:24:06] INFO  No tools returned via semantic search; trying next lookup strategy.
[09:24:06] OK    Loaded 4 Composio tool(s) via exact email tool slugs
[09:24:06] INFO  Selected tool: GMAIL_CREATE_EMAIL_DRAFT
[09:24:06] OK    Using model: gpt-5.4-mini
[09:24:06] INFO  Available: 1 tool(s), 0 resource(s).
[09:24:06] STEP  Preparing prompt & JSON schema &
[09:24:06] STEP  Calling model: gpt-5.4-mini &
[09:24:09] OK    Model responded in 3.21s.
[09:24:09] INFO  Raw content length: 1567 chars.
[09:24:09] OK    Parsed 5 scenario(s) successfully.
[09:24:09] INFO  Difficulty distribution:
   1/5:   1  █
   2/5:   1  █
   3/5:   1  █
   4/5:   1  █
   5/5:   1  █
   1. Draft a polite follow-up email to a client asking for an update on the proposal review, and include a brief summary of w…  (difficulty 1/5)
   2. Create an email draft to your manager summarizing the status of this week’s project milestones, with CC to two teammates…  (difficulty 2/5)
   3. Prepare a reply draft in an existin

In [ ]:
# @title 从 Composio 收集工具
search_tools = []
all_tools = {}

# 收集与当前 USE_CASE 匹配的 Composio 工具
if composio is not None:
    try:
        tools_result, resources_result = await list_composio_tools_and_resources()
        composio_tools = []
        for tool in tools_result:
            tool_dict = composio_tool_to_art_info(tool)
            tool_dict["source"] = "composio"
            search_tools.append(tool_dict)
            composio_tools.append(tool_dict)
        all_tools["composio"] = composio_tools
        if composio_tools:
            ok(f"Collected {len(composio_tools)} tool(s) from Composio")
    except Exception as e:
        warn(f"Failed to collect Composio tools: {e}")
        all_tools["composio"] = []

if search_tools:
    info(f"Total tools collected: {len(search_tools)} from {len(all_tools)} source(s)")
else:
    warn("No tools collected after all Composio lookup strategies.")
    warn("If you do not see lookup-strategy messages above, rerun the Composio Helper Functions cell first.")

[09:24:11] INFO  No tools returned via semantic search; trying next lookup strategy.
[09:24:12] OK    Loaded 4 Composio tool(s) via exact email tool slugs
[09:24:12] OK    Collected 4 tool(s) from Composio
[09:24:12] INFO  Total tools collected: 4 from 1 source(s)


In [ ]:
# @title 使用增强工具集生成场景

async def generate_scenarios_with_enhanced_tools(search_tools: List[Dict], num_scenarios: int = 15):
    """
    生成能够利用增强工具集的场景。
    这些场景应鼓励智能体有效组合多个工具。
    """
    if not OPENROUTER_API_KEY:
        warn("OPENROUTER_API_KEY required for scenario generation")
        return []

    # 按来源整理工具，供场景生成使用
    tools_by_source = {}
    for tool in search_tools:
        source = tool["source"]
        if source not in tools_by_source:
            tools_by_source[source] = []
        tools_by_source[source].append({
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": tool.get("parameters", {})
        })

    info(f"Generating {num_scenarios} scenarios with enhanced tool set...")
    info(f"Tools available from: {', '.join(tools_by_source.keys())}")

    # 合并全部工具，用于生成场景
    all_tools_flat = [
        {
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": tool.get("parameters", {})
        }
        for tool in search_tools
    ]

    try:
        scenario_collection = await generate_scenarios(
            tools=all_tools_flat,
            resources=[],  # 如有可用 resources，可在此加入
            num_scenarios=num_scenarios,
            show_preview=False,
            generator_model=LLM_MODEL,
            generator_api_key=OPENROUTER_API_KEY,
        )

        enhanced_scenarios = [
            {
                "task": s.task,
                "difficulty": s.difficulty,
                "tools_available": len(all_tools_flat)
            }
            for s in scenario_collection.scenarios
        ]

        ok(f"Generated {len(enhanced_scenarios)} enhanced scenarios")

        info("\nSample enhanced scenarios (encouraging multi-tool usage):")
        preview_scenarios(enhanced_scenarios, n=min(5, len(enhanced_scenarios)))

        return enhanced_scenarios

    except Exception as e:
        warn(f"Scenario generation failed: {e}")
        return []

# 生成增强场景
if search_tools and OPENROUTER_API_KEY:
    enhanced_scenarios = await generate_scenarios_with_enhanced_tools(
        search_tools,
        num_scenarios=15
    )


[09:24:12] INFO  Generating 15 scenarios with enhanced tool set...
[09:24:12] INFO  Tools available from: composio
[09:24:12] OK    Using model: gpt-5.4-mini
[09:24:12] INFO  Available: 4 tool(s), 0 resource(s).
[09:24:12] STEP  Preparing prompt & JSON schema &
[09:24:12] STEP  Calling model: gpt-5.4-mini &
[09:24:27] OK    Model responded in 15.51s.
[09:24:27] INFO  Raw content length: 3561 chars.
[09:24:27] OK    Parsed 15 scenario(s) successfully.
[09:24:27] INFO  Difficulty distribution:
   1/5:   2  ██
   2/5:   2  ██
   3/5:   3  ███
   4/5:   4  ████
   5/5:   4  ████
[09:24:27] OK    Generated 15 scenarios in 15.58s total.
[09:24:27] OK    Generated 15 enhanced scenarios
[09:24:27] INFO  
Sample enhanced scenarios (encouraging multi-tool usage):
   1. Draft a polite follow-up email to a recruiter thanking them for the interview and asking about next steps, then summariz&  (difficulty 1/5)
   2. Send a same-day meeting reminder to a small project team with the agenda and Zoom li

### 后续可以进一步尝试的方向：

1. **增加比较样本（More Comparisons）** ✅
   - 每个场景生成 5–10 条以上轨迹，而不只是 2–3 条
   - 使用 RULER 对轨迹进行相对排序
   - 改变策略，例如使用不同模型、Prompt 或工具使用模式
   - 比较越充分，模型获得的学习信号通常越丰富

2. **扩展工具能力（Better Tools）** ✅
   - 将 Composio 作为大型动态工具生态接入
   - 针对每个用例选择互补的工具和工具包
   - 生成能够鼓励多工具协同使用的场景
   - 训练智能体更合理地组合不同工具的结果

**请记住**：RULER 会从你实际使用的工具和场景中学习什么叫“好”，不需要预先准备人工标注数据。

### 更多资源（Additional Resources）

- **ART Documentation**: https://art.openpipe.ai
- **RULER Guide**: https://art.openpipe.ai/fundamentals/ruler
- **Composio SDK**: https://github.com/composiohq/composio
- **Composio Docs**: https://docs.composio.dev/
